In [ ]:
import torch
import pandas as pd
from tqdm import tqdm

from matplotlib import pyplot as plt
from matplotlib import ticker 

torch.manual_seed(42)

from dataclass import dataset

$$
\begin{array}{|c|l|c|c|}
\hline
 & \text{Reaction} & \text{Stoichiometry} & \text{Rate} \\
\hline
\text{Colonization} & \emptyset \to E & +1 & \alpha \\
\hline
\text{Replication} & E \to 2E & +1 & \mu E \\
\hline
\text{Competition} & E \to \emptyset & -1 & \left(\frac{\mu}{K}\right) E^2 \\
\hline
\text{Expulsion} & E \to \emptyset & -1 & dE \\
\hline
\end{array}
$$


In [ ]:
# Loading the synthetic dataset
dseed = 0
df = pd.read_csv('synthetic_data/synth_seed{}.csv'.format(dseed))
df

In [ ]:
#Loading the ground truth

gt = pd.read_csv('synthetic_data/gt_map.csv', header=None, delim_whitespace=True).to_numpy()
gt = torch.tensor(gt[gt[:,0]==dseed][0,1:]).float()
gt

In [ ]:
# The dataset need to be send to the `dataset` class
Ts     = df['Time'].to_numpy()
counts = df['Counts'].to_numpy()
dils   = df['Dilution'].to_numpy()
data = dataset(Ts,counts,dils)

In [ ]:
#Define log posterior and check up in the lp_gt
value = torch.tensor( (1/20, 1/4, 2*1e5, 1e-3) ).to(data.device)
prior = torch.distributions.LogNormal(torch.log(value),torch.ones_like(value)*1/3)
logposterior = lambda value,data: data.loglike(value) + prior.log_prob(value).sum()

lp_gt = logposterior(gt.to(data.device),data)

In [ ]:
def proposal(th,S_prop=1e-2):
    lth = torch.log(th)
    lprop = lth + torch.rand_like(th)*S_prop
    return torch.exp(lprop)

def next_MCMC_sample(params,lp):
    lp = logposterior(params,data)

    params_prop = proposal(params)
    lp_prop = logposterior(params_prop,data)

    if torch.log(torch.rand(1)).item() < (lp_prop-lp).item():
        lp = lp_prop
        params = params_prop
    
    return params,lp
    
params = prior.sample()
lp = -torch.inf


In [ ]:
N_burn = 300
N_mcmc = 1000

In [ ]:
mcmc_params =[]
mcmc_lps = []

In [ ]:
for s in tqdm(range(N_burn+N_mcmc)):
    if (s+1)%10 == 0:
        print(s+1, 'lp', lp.item())
        print(params.cpu().numpy())

    params,lp = next_MCMC_sample(params,lp)
    mcmc_params.append(params*1)
    mcmc_lps.append(lp.item())

In [ ]:
posterior_samples = torch.stack(mcmc_params[-N_mcmc:]).cpu().numpy()

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Colonization rate (/h)','Replication rate (/h)','Capacity','Death rate (/h)']
for i in range(4):
    ax[i].hist(posterior_samples[:, i], bins=30, density=True, alpha=0.7)
    ax[i].set_xlabel(titles[i])
    ax[i].axvline(gt[i].item(),color='r')

formatter = ticker.ScalarFormatter(useMathText=True)
formatter.set_scientific(True)

ax[0].set_ylabel('Density')
fig.tight_layout()
plt.show()

In [ ]:
plt.plot(mcmc_lps)

In [ ]:
torch.rand((4,10))